In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from PIL import Image
from tqdm import tqdm

# =========================
# CONFIG
# =========================
DATASET_PATH = "your_dataset_path_here"  # root folder containing all animal folders
IMG_SIZE = (64, 64)  # resize images
TEST_SIZE = 0.2  # 80% train, 20% test
K = 3  # neighbors

# =========================
# LOAD DATA
# =========================
images = []
labels = []

print("Loading images...")

for label in os.listdir(DATASET_PATH):
    class_path = os.path.join(DATASET_PATH, label)

    if not os.path.isdir(class_path):
        continue

    for img_name in tqdm(os.listdir(class_path), desc=f"Processing {label}"):
        img_path = os.path.join(class_path, img_name)

        try:
            img = Image.open(img_path).convert("RGB")
            img = img.resize(IMG_SIZE)
            img = np.array(img)
            img = img.flatten()  # convert to 1D vector

            images.append(img)
            labels.append(label)

        except:
            continue

X = np.array(images)
y = np.array(labels)

print(f"Total samples: {len(X)}")

# =========================
# ENCODE LABELS
# =========================
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# =========================
# TRAIN-TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=TEST_SIZE, stratify=y_encoded, random_state=42
)

print(f"Train samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# =========================
# TRAIN KNN
# =========================
print("Training KNN...")

knn = KNeighborsClassifier(n_neighbors=K, n_jobs=-1)
knn.fit(X_train, y_train)

# =========================
# TEST MODEL
# =========================
print("Testing model...")

y_pred = knn.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"\n✅ Accuracy: {accuracy * 100:.2f}%")